In [ ]:
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

REPO_ROOT = Path("/content/adversarial-backdoor-neural-cleanse")

if not REPO_ROOT.exists():
    raise FileNotFoundError(
        "Clone the repository to /content/adversarial-backdoor-neural-cleanse first."
    )

os.chdir(REPO_ROOT)

SEED = 14
DATA_ROOT = REPO_ROOT / "data"
OUTPUT_ROOT = REPO_ROOT / "outputs"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
NC_ROOT = OUTPUT_ROOT / "neural_cleanse"
FIGURE_ROOT = OUTPUT_ROOT / "figures"

NUM_CLASSES = 10
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

VAL_SIZE = 5000
BATCH_SIZE = 128
NUM_WORKERS = 2

EPOCHS = 50
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LR_MILESTONES = [30, 40]
LR_GAMMA = 0.1

POISON_RATE = 0.10
TARGET_CLASS = 0
TRIGGER_SIZE = 4
TRIGGER_VALUE = 1.0

PGD_TRAIN_EPSILON = 8 / 255
PGD_TRAIN_ALPHA = 2 / 255
PGD_TRAIN_STEPS = 7

PGD_EVAL_EPSILON = 8 / 255
PGD_EVAL_ALPHA = 2 / 255
PGD_EVAL_STEPS = 20

NC_STEPS = 1000
NC_LEARNING_RATE = 0.1
NC_INITIAL_COST = 1e-3
NC_COST_MULTIPLIER = 1.5
NC_COST_PATIENCE = 50
NC_OPT_SUCCESS_THRESHOLD = 0.99
NC_DETECTION_SUCCESS_THRESHOLD = 0.95
NC_ANOMALY_THRESHOLD = 2.0

AUTO_DOWNLOAD_CHECKPOINTS = True
SKIP_EXISTING_FINAL_CHECKPOINTS = False

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

for directory in [
    CHECKPOINT_ROOT,
    METRICS_ROOT,
    NC_ROOT,
    FIGURE_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Working directory:", Path.cwd())
print("Device:", DEVICE)

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def create_cifar10_datasets(
    root,
    val_size=VAL_SIZE,
    seed=SEED,
    download=True,
):
    root = Path(root)

    train_transform = transforms.Compose(
        [
            transforms.RandomCrop(
                32,
                padding=4,
            ),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
        ]
    )

    eval_transform = transforms.ToTensor()

    train_full = datasets.CIFAR10(
        root=root,
        train=True,
        transform=train_transform,
        download=download,
    )

    val_full = datasets.CIFAR10(
        root=root,
        train=True,
        transform=eval_transform,
        download=False,
    )

    test_dataset = datasets.CIFAR10(
        root=root,
        train=False,
        transform=eval_transform,
        download=download,
    )

    if not 0 < val_size < len(train_full):
        raise ValueError(
            f"val_size must be between 1 and {len(train_full) - 1}."
        )

    generator = torch.Generator().manual_seed(seed)

    indices = torch.randperm(
        len(train_full),
        generator=generator,
    )

    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    train_dataset = Subset(
        train_full,
        train_indices.tolist(),
    )

    val_dataset = Subset(
        val_full,
        val_indices.tolist(),
    )

    return (
        train_dataset,
        val_dataset,
        test_dataset,
    )


def make_loader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED,
):
    generator = torch.Generator().manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        generator=generator if shuffle else None,
    )


def get_label(
    dataset,
    index,
):
    if isinstance(
        dataset,
        Subset,
    ):
        original_index = dataset.indices[index]

        return get_label(
            dataset.dataset,
            original_index,
        )

    if hasattr(
        dataset,
        "targets",
    ):
        return int(
            dataset.targets[index]
        )

    _, label = dataset[index]

    return int(label)


def create_square_trigger(
    channels=3,
    trigger_size=TRIGGER_SIZE,
    value=TRIGGER_VALUE,
):
    if trigger_size <= 0:
        raise ValueError(
            "trigger_size must be positive."
        )

    if not 0.0 <= value <= 1.0:
        raise ValueError(
            "trigger value must be in [0, 1]."
        )

    return torch.full(
        (
            channels,
            trigger_size,
            trigger_size,
        ),
        fill_value=value,
        dtype=torch.float32,
    )


def apply_trigger(
    image,
    trigger,
):
    if image.ndim != 3:
        raise ValueError(
            "image must have shape [C, H, W]."
        )

    if trigger.ndim != 3:
        raise ValueError(
            "trigger must have shape [C, H, W]."
        )

    channels, height, width = image.shape
    trigger_channels, trigger_height, trigger_width = trigger.shape

    if channels != trigger_channels:
        raise ValueError(
            "image and trigger channels must match."
        )

    if (
        trigger_height > height
        or trigger_width > width
    ):
        raise ValueError(
            "trigger cannot be larger than the image."
        )

    poisoned_image = image.clone()

    poisoned_image[
        :,
        height - trigger_height : height,
        width - trigger_width : width,
    ] = trigger.to(
        device=image.device,
        dtype=image.dtype,
    )

    return poisoned_image


class PoisonedDataset(Dataset):

    def __init__(
        self,
        dataset,
        trigger,
        poison_rate=POISON_RATE,
        target_class=TARGET_CLASS,
        seed=SEED,
        exclude_target_class=True,
    ):
        if not 0.0 <= poison_rate <= 1.0:
            raise ValueError(
                "poison_rate must be in [0, 1]."
            )

        self.dataset = dataset
        self.trigger = trigger
        self.target_class = target_class

        candidate_indices = []

        for index in range(
            len(dataset)
        ):
            label = get_label(
                dataset,
                index,
            )

            if (
                exclude_target_class
                and label == target_class
            ):
                continue

            candidate_indices.append(
                index
            )

        number_of_poisoned_samples = min(
            int(
                len(dataset)
                * poison_rate
            ),
            len(candidate_indices),
        )

        rng = random.Random(
            seed
        )

        self.poison_indices = set(
            rng.sample(
                candidate_indices,
                number_of_poisoned_samples,
            )
        )

    def __len__(self):
        return len(
            self.dataset
        )

    def __getitem__(
        self,
        index,
    ):
        image, label = self.dataset[index]

        if index in self.poison_indices:
            image = apply_trigger(
                image,
                self.trigger,
            )

            label = self.target_class

        return (
            image,
            label,
        )


class BackdoorTestDataset(Dataset):

    def __init__(
        self,
        dataset,
        trigger,
        target_class=TARGET_CLASS,
    ):
        self.dataset = dataset
        self.trigger = trigger
        self.target_class = target_class

        self.indices = [
            index
            for index in range(
                len(dataset)
            )
            if get_label(
                dataset,
                index,
            ) != target_class
        ]

    def __len__(self):
        return len(
            self.indices
        )

    def __getitem__(
        self,
        index,
    ):
        original_index = self.indices[
            index
        ]

        image, _ = self.dataset[
            original_index
        ]

        image = apply_trigger(
            image,
            self.trigger,
        )

        return (
            image,
            self.target_class,
        )


set_seed()

train_dataset, val_dataset, test_dataset = (
    create_cifar10_datasets(
        root=DATA_ROOT,
    )
)

trigger = create_square_trigger()

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Trigger shape:", tuple(trigger.shape))

In [ ]:
CIFAR10_MEAN = (
    0.4914,
    0.4822,
    0.4465,
)

CIFAR10_STD = (
    0.2470,
    0.2435,
    0.2616,
)


class Normalize(nn.Module):

    def __init__(
        self,
        mean=CIFAR10_MEAN,
        std=CIFAR10_STD,
    ):
        super().__init__()

        mean = torch.tensor(
            mean,
            dtype=torch.float32,
        ).view(
            1,
            3,
            1,
            1,
        )

        std = torch.tensor(
            std,
            dtype=torch.float32,
        ).view(
            1,
            3,
            1,
            1,
        )

        self.register_buffer(
            "mean",
            mean,
        )

        self.register_buffer(
            "std",
            std,
        )

    def forward(
        self,
        x,
    ):
        return (
            x - self.mean
        ) / self.std


class BasicBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            out_channels
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        self.bn2 = nn.BatchNorm2d(
            out_channels
        )

        if (
            stride != 1
            or in_channels != out_channels
        ):
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(
                    out_channels
                ),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(
        self,
        x,
    ):
        identity = self.shortcut(
            x
        )

        out = self.conv1(
            x
        )

        out = self.bn1(
            out
        )

        out = self.relu(
            out
        )

        out = self.conv2(
            out
        )

        out = self.bn2(
            out
        )

        out = (
            out
            + identity
        )

        out = self.relu(
            out
        )

        return out


class ResNet20(nn.Module):

    def __init__(
        self,
        num_classes=NUM_CLASSES,
    ):
        super().__init__()

        self.normalize = Normalize()

        self.in_channels = 16

        self.conv1 = nn.Conv2d(
            3,
            16,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

        self.bn1 = nn.BatchNorm2d(
            16
        )

        self.relu = nn.ReLU(
            inplace=True
        )

        self.layer1 = self._make_layer(
            out_channels=16,
            num_blocks=3,
            stride=1,
        )

        self.layer2 = self._make_layer(
            out_channels=32,
            num_blocks=3,
            stride=2,
        )

        self.layer3 = self._make_layer(
            out_channels=64,
            num_blocks=3,
            stride=2,
        )

        self.avg_pool = nn.AdaptiveAvgPool2d(
            (
                1,
                1,
            )
        )

        self.fc = nn.Linear(
            64,
            num_classes,
        )

        self._initialize_weights()

    def _make_layer(
        self,
        out_channels,
        num_blocks,
        stride,
    ):
        strides = [
            stride
        ] + [
            1
        ] * (
            num_blocks - 1
        )

        blocks = []

        for block_stride in strides:
            blocks.append(
                BasicBlock(
                    in_channels=self.in_channels,
                    out_channels=out_channels,
                    stride=block_stride,
                )
            )

            self.in_channels = out_channels

        return nn.Sequential(
            *blocks
        )

    def _initialize_weights(
        self,
    ):
        for module in self.modules():

            if isinstance(
                module,
                nn.Conv2d,
            ):
                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_out",
                    nonlinearity="relu",
                )

            elif isinstance(
                module,
                nn.BatchNorm2d,
            ):
                nn.init.constant_(
                    module.weight,
                    1,
                )

                nn.init.constant_(
                    module.bias,
                    0,
                )

    def forward(
        self,
        x,
    ):
        x = self.normalize(
            x
        )

        x = self.conv1(
            x
        )

        x = self.bn1(
            x
        )

        x = self.relu(
            x
        )

        x = self.layer1(
            x
        )

        x = self.layer2(
            x
        )

        x = self.layer3(
            x
        )

        x = self.avg_pool(
            x
        )

        x = torch.flatten(
            x,
            1,
        )

        return self.fc(
            x
        )


def create_classifier():
    return ResNet20(
        num_classes=NUM_CLASSES,
    )


model_check = create_classifier().to(
    DEVICE
)

with torch.no_grad():
    shape_check = model_check(
        torch.zeros(
            2,
            3,
            32,
            32,
            device=DEVICE,
        )
    ).shape

print("Model output shape:", tuple(shape_check))

del model_check

In [ ]:
def pgd_linf(
    model,
    images,
    labels,
    epsilon,
    alpha,
    steps,
    random_start=True,
):
    original_images = images.detach()

    if random_start:
        perturbation = torch.empty_like(
            original_images
        ).uniform_(
            -epsilon,
            epsilon,
        )

        images_adv = torch.clamp(
            original_images
            + perturbation,
            0.0,
            1.0,
        )
    else:
        images_adv = original_images.clone()

    was_training = model.training

    model.eval()

    try:
        for _ in range(
            steps
        ):
            images_adv = (
                images_adv
                .detach()
            )

            images_adv.requires_grad_(
                True
            )

            logits = model(
                images_adv
            )

            loss = F.cross_entropy(
                logits,
                labels,
            )

            gradient = torch.autograd.grad(
                loss,
                images_adv,
                only_inputs=True,
            )[0]

            with torch.no_grad():
                images_adv = (
                    images_adv
                    + alpha
                    * gradient.sign()
                )

                perturbation = (
                    images_adv
                    - original_images
                )

                perturbation = torch.clamp(
                    perturbation,
                    -epsilon,
                    epsilon,
                )

                images_adv = torch.clamp(
                    original_images
                    + perturbation,
                    0.0,
                    1.0,
                )

    finally:
        model.train(
            was_training
        )

    return images_adv.detach()


def train_one_epoch(
    model,
    loader,
    optimizer,
    adversarial=False,
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        if adversarial:
            images = pgd_linf(
                model=model,
                images=images,
                labels=labels,
                epsilon=PGD_TRAIN_EPSILON,
                alpha=PGD_TRAIN_ALPHA,
                steps=PGD_TRAIN_STEPS,
                random_start=True,
            )

            model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            images
        )

        loss = F.cross_entropy(
            logits,
            labels,
        )

        loss.backward()

        optimizer.step()

        batch_size = labels.size(
            0
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_correct += (
            logits.argmax(
                dim=1
            )
            == labels
        ).sum().item()

        total_samples += batch_size

    return {
        "loss":
            total_loss
            / total_samples,

        "accuracy":
            total_correct
            / total_samples,
    }


@torch.no_grad()
def evaluate_accuracy(
    model,
    loader,
):
    model.eval()

    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            images
        )

        total_correct += (
            logits.argmax(
                dim=1
            )
            == labels
        ).sum().item()

        total_samples += labels.size(
            0
        )

    return (
        total_correct
        / total_samples
    )


def evaluate_adversarial_accuracy(
    model,
    loader,
):
    model.eval()

    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        images_adv = pgd_linf(
            model=model,
            images=images,
            labels=labels,
            epsilon=PGD_EVAL_EPSILON,
            alpha=PGD_EVAL_ALPHA,
            steps=PGD_EVAL_STEPS,
            random_start=True,
        )

        with torch.no_grad():
            logits = model(
                images_adv
            )

        total_correct += (
            logits.argmax(
                dim=1
            )
            == labels
        ).sum().item()

        total_samples += labels.size(
            0
        )

    return (
        total_correct
        / total_samples
    )


@torch.no_grad()
def evaluate_backdoor_asr(
    model,
    loader,
):
    model.eval()

    total_success = 0
    total_samples = 0

    for images, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            images
        )

        predictions = logits.argmax(
            dim=1
        )

        total_success += (
            predictions
            == TARGET_CLASS
        ).sum().item()

        total_samples += images.size(
            0
        )

    return (
        total_success
        / total_samples
    )


def download_file(
    path,
):
    if not AUTO_DOWNLOAD_CHECKPOINTS:
        return

    try:
        from google.colab import files
        files.download(
            str(path)
        )
    except ImportError:
        print(
            "Automatic download is only available in Colab:",
            path,
        )


def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    epoch,
    val_accuracy,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "val_accuracy":
                val_accuracy,
        },
        path,
    )


def load_checkpoint_model(
    path,
):
    model = create_classifier().to(
        DEVICE
    )

    checkpoint = torch.load(
        path,
        map_location=DEVICE,
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.eval()

    return model


def train_experiment(
    model_name,
    train_dataset,
    adversarial=False,
):
    set_seed(
        SEED
    )

    final_path = (
        CHECKPOINT_ROOT
        / f"{model_name}_final.pt"
    )

    best_path = (
        CHECKPOINT_ROOT
        / f"{model_name}_best.pt"
    )

    if (
        SKIP_EXISTING_FINAL_CHECKPOINTS
        and final_path.exists()
    ):
        print(
            "Skipping existing checkpoint:",
            final_path,
        )

        return final_path

    train_loader = make_loader(
        train_dataset,
        shuffle=True,
        seed=SEED,
    )

    val_loader = make_loader(
        val_dataset,
        shuffle=False,
    )

    model = create_classifier().to(
        DEVICE
    )

    optimizer = optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=LR_MILESTONES,
        gamma=LR_GAMMA,
    )

    best_accuracy = -1.0
    history = []

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            adversarial=adversarial,
        )

        val_accuracy = evaluate_accuracy(
            model,
            val_loader,
        )

        current_lr = optimizer.param_groups[
            0
        ]["lr"]

        history.append(
            {
                "epoch":
                    epoch,

                "lr":
                    current_lr,

                "train_loss":
                    train_metrics[
                        "loss"
                    ],

                "train_accuracy":
                    train_metrics[
                        "accuracy"
                    ],

                "val_accuracy":
                    val_accuracy,
            }
        )

        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy

            save_checkpoint(
                path=best_path,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                val_accuracy=val_accuracy,
            )

        scheduler.step()

        save_checkpoint(
            path=final_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            val_accuracy=val_accuracy,
        )

        print(
            f"{model_name} | "
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"LR {current_lr:.5f} | "
            f"Train Loss {train_metrics['loss']:.4f} | "
            f"Train Acc {train_metrics['accuracy']:.4f} | "
            f"Val Acc {val_accuracy:.4f}"
        )

    history_path = (
        METRICS_ROOT
        / f"{model_name}_training_history.csv"
    )

    pd.DataFrame(
        history
    ).to_csv(
        history_path,
        index=False,
    )

    print(
        "Saved:",
        final_path
    )

    print(
        "Best validation accuracy:",
        best_accuracy
    )

    download_file(
        final_path
    )

    return final_path

In [ ]:
clean_checkpoint = train_experiment(
    model_name="clean",
    train_dataset=train_dataset,
    adversarial=False,
)

In [ ]:
poisoned_train_dataset = PoisonedDataset(
    dataset=train_dataset,
    trigger=trigger,
    poison_rate=POISON_RATE,
    target_class=TARGET_CLASS,
    seed=SEED,
)

print(
    "Poisoned samples:",
    len(
        poisoned_train_dataset.poison_indices
    ),
)

badnet_checkpoint = train_experiment(
    model_name="badnet",
    train_dataset=poisoned_train_dataset,
    adversarial=False,
)

In [ ]:
adv_badnet_checkpoint = train_experiment(
    model_name="adv_badnet",
    train_dataset=poisoned_train_dataset,
    adversarial=True,
)

In [ ]:
test_loader = make_loader(
    test_dataset,
    shuffle=False,
)

backdoor_test_dataset = BackdoorTestDataset(
    dataset=test_dataset,
    trigger=trigger,
    target_class=TARGET_CLASS,
)

backdoor_test_loader = make_loader(
    backdoor_test_dataset,
    shuffle=False,
)

CHECKPOINTS = {
    "clean":
        clean_checkpoint,

    "badnet":
        badnet_checkpoint,

    "adv_badnet":
        adv_badnet_checkpoint,
}

evaluation_rows = []

for model_name, checkpoint_path in CHECKPOINTS.items():
    model = load_checkpoint_model(
        checkpoint_path
    )

    clean_accuracy = evaluate_accuracy(
        model,
        test_loader,
    )

    backdoor_asr = evaluate_backdoor_asr(
        model,
        backdoor_test_loader,
    )

    robust_accuracy = evaluate_adversarial_accuracy(
        model,
        test_loader,
    )

    evaluation_rows.append(
        {
            "model":
                model_name,

            "clean_accuracy":
                clean_accuracy,

            "backdoor_asr":
                backdoor_asr,

            "pgd20_robust_accuracy":
                robust_accuracy,
        }
    )

    print(
        f"{model_name} | "
        f"Clean {clean_accuracy:.4f} | "
        f"ASR {backdoor_asr:.4f} | "
        f"PGD-20 {robust_accuracy:.4f}"
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

evaluation_df = pd.DataFrame(
    evaluation_rows
)

evaluation_path = (
    METRICS_ROOT
    / "model_metrics.csv"
)

evaluation_df.to_csv(
    evaluation_path,
    index=False,
)

display(
    evaluation_df
)

In [ ]:
class NeuralCleanseTrigger(nn.Module):

    def __init__(
        self,
        channels=3,
        height=32,
        width=32,
    ):
        super().__init__()

        self.mask_param = nn.Parameter(
            torch.zeros(
                1,
                1,
                height,
                width,
            )
        )

        self.pattern_param = nn.Parameter(
            torch.zeros(
                1,
                channels,
                height,
                width,
            )
        )

    def get_mask(
        self,
    ):
        return torch.sigmoid(
            self.mask_param
        )

    def get_pattern(
        self,
    ):
        return torch.sigmoid(
            self.pattern_param
        )

    def forward(
        self,
        images,
    ):
        mask = self.get_mask()
        pattern = self.get_pattern()

        poisoned = (
            (1.0 - mask)
            * images
            + mask
            * pattern
        )

        return (
            poisoned,
            mask,
            pattern,
        )


@torch.no_grad()
def evaluate_reconstructed_trigger(
    model,
    loader,
    target_class,
    mask,
    pattern,
):
    model.eval()

    total_success = 0
    total_samples = 0

    mask = mask.to(
        DEVICE
    )

    pattern = pattern.to(
        DEVICE
    )

    for images, labels in loader:
        keep = (
            labels
            != target_class
        )

        if not bool(
            keep.any()
        ):
            continue

        images = images[
            keep
        ].to(
            DEVICE,
            non_blocking=True,
        )

        poisoned = (
            (1.0 - mask)
            * images
            + mask
            * pattern
        )

        predictions = model(
            poisoned
        ).argmax(
            dim=1
        )

        total_success += (
            predictions
            == target_class
        ).sum().item()

        total_samples += images.size(
            0
        )

    if total_samples == 0:
        return 0.0

    return (
        total_success
        / total_samples
    )


def reverse_engineer_trigger(
    model,
    loader,
    target_class,
):
    model.eval()

    trigger_model = NeuralCleanseTrigger().to(
        DEVICE
    )

    optimizer = optim.Adam(
        trigger_model.parameters(),
        lr=NC_LEARNING_RATE,
    )

    cost = NC_INITIAL_COST
    success_streak = 0
    failure_streak = 0

    best_mask = None
    best_pattern = None
    best_mask_norm = float(
        "inf"
    )

    loader_iterator = iter(
        loader
    )

    for step in range(
        1,
        NC_STEPS + 1,
    ):
        try:
            images, labels = next(
                loader_iterator
            )

        except StopIteration:
            loader_iterator = iter(
                loader
            )

            images, labels = next(
                loader_iterator
            )

        keep = (
            labels
            != target_class
        )

        if not bool(
            keep.any()
        ):
            continue

        images = images[
            keep
        ].to(
            DEVICE,
            non_blocking=True,
        )

        targets = torch.full(
            (
                images.size(0),
            ),
            target_class,
            dtype=torch.long,
            device=DEVICE,
        )

        poisoned, mask, pattern = trigger_model(
            images
        )

        logits = model(
            poisoned
        )

        loss_ce = F.cross_entropy(
            logits,
            targets,
        )

        loss_mask = mask.abs().sum()

        loss = (
            loss_ce
            + cost
            * loss_mask
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            success = (
                logits.argmax(
                    dim=1
                )
                == targets
            ).float().mean().item()

            mask_norm = (
                trigger_model
                .get_mask()
                .abs()
                .sum()
                .item()
            )

            if (
                success
                >= NC_OPT_SUCCESS_THRESHOLD
                and mask_norm
                < best_mask_norm
            ):
                best_mask_norm = mask_norm

                best_mask = (
                    trigger_model
                    .get_mask()
                    .detach()
                    .cpu()
                    .clone()
                )

                best_pattern = (
                    trigger_model
                    .get_pattern()
                    .detach()
                    .cpu()
                    .clone()
                )

        if success >= NC_OPT_SUCCESS_THRESHOLD:
            success_streak += 1
            failure_streak = 0
        else:
            failure_streak += 1
            success_streak = 0

        if success_streak >= NC_COST_PATIENCE:
            cost = min(
                cost
                * NC_COST_MULTIPLIER,
                100.0,
            )

            success_streak = 0

        if failure_streak >= NC_COST_PATIENCE:
            cost = max(
                cost
                / NC_COST_MULTIPLIER,
                1e-8,
            )

            failure_streak = 0

        if (
            step == 1
            or step % 100 == 0
            or step == NC_STEPS
        ):
            print(
                f"Class {target_class} | "
                f"Step {step:04d}/{NC_STEPS} | "
                f"Success {success:.4f} | "
                f"Mask {mask_norm:.4f} | "
                f"Cost {cost:.6f}"
            )

    if best_mask is None:
        best_mask = (
            trigger_model
            .get_mask()
            .detach()
            .cpu()
        )

        best_pattern = (
            trigger_model
            .get_pattern()
            .detach()
            .cpu()
        )

        best_mask_norm = (
            best_mask
            .abs()
            .sum()
            .item()
        )

    reconstructed_asr = evaluate_reconstructed_trigger(
        model=model,
        loader=loader,
        target_class=target_class,
        mask=best_mask,
        pattern=best_pattern,
    )

    return {
        "target_class":
            target_class,

        "mask":
            best_mask,

        "pattern":
            best_pattern,

        "mask_norm":
            float(
                best_mask_norm
            ),

        "reconstructed_asr":
            float(
                reconstructed_asr
            ),

        "final_cost":
            float(
                cost
            ),
    }


def neural_cleanse_anomaly_indices(
    mask_norms,
):
    values = np.asarray(
        list(
            mask_norms.values()
        ),
        dtype=np.float64,
    )

    median = float(
        np.median(
            values
        )
    )

    mad = float(
        np.median(
            np.abs(
                values
                - median
            )
        )
    )

    denominator = max(
        1.4826
        * mad,
        1e-12,
    )

    scores = {}

    for class_id, value in mask_norms.items():
        scores[
            class_id
        ] = max(
            0.0,
            (
                median
                - value
            )
            / denominator,
        )

    return {
        "median":
            median,

        "mad":
            mad,

        "scores":
            scores,
    }


nc_loader = make_loader(
    val_dataset,
    shuffle=True,
    seed=SEED,
)

In [ ]:
all_nc_rows = []
detection_rows = []

for model_name, checkpoint_path in CHECKPOINTS.items():
    print()
    print(
        "Running Neural Cleanse on:",
        model_name
    )

    set_seed(
        SEED
    )

    model = load_checkpoint_model(
        checkpoint_path
    )

    model_dir = (
        NC_ROOT
        / model_name
    )

    model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    class_results = {}

    for target_class in range(
        NUM_CLASSES
    ):
        result = reverse_engineer_trigger(
            model=model,
            loader=nc_loader,
            target_class=target_class,
        )

        class_results[
            target_class
        ] = result

        torch.save(
            result,
            model_dir
            / f"class_{target_class}.pt",
        )

        print(
            f"{model_name} | "
            f"Class {target_class} | "
            f"Mask {result['mask_norm']:.4f} | "
            f"Recovered ASR {result['reconstructed_asr']:.4f}"
        )

    mask_norms = {
        class_id:
            result[
                "mask_norm"
            ]
        for class_id, result
        in class_results.items()
    }

    anomaly = neural_cleanse_anomaly_indices(
        mask_norms
    )

    model_rows = []

    for class_id, result in class_results.items():
        row = {
            "model":
                model_name,

            "target_class":
                class_id,

            "class_name":
                CLASS_NAMES[
                    class_id
                ],

            "mask_norm":
                result[
                    "mask_norm"
                ],

            "reconstructed_asr":
                result[
                    "reconstructed_asr"
                ],

            "anomaly_score":
                anomaly[
                    "scores"
                ][
                    class_id
                ],
        }

        model_rows.append(
            row
        )

        all_nc_rows.append(
            row
        )

    model_summary = pd.DataFrame(
        model_rows
    )

    model_summary.to_csv(
        model_dir
        / "summary.csv",
        index=False,
    )

    suspected_class = max(
        anomaly[
            "scores"
        ],
        key=anomaly[
            "scores"
        ].get,
    )

    suspected_result = class_results[
        suspected_class
    ]

    suspected_score = anomaly[
        "scores"
    ][
        suspected_class
    ]

    detected = (
        suspected_score
        >= NC_ANOMALY_THRESHOLD
        and suspected_result[
            "reconstructed_asr"
        ]
        >= NC_DETECTION_SUCCESS_THRESHOLD
    )

    detection_rows.append(
        {
            "model":
                model_name,

            "suspected_class":
                suspected_class,

            "suspected_class_name":
                CLASS_NAMES[
                    suspected_class
                ],

            "anomaly_score":
                suspected_score,

            "reconstructed_asr":
                suspected_result[
                    "reconstructed_asr"
                ],

            "median_mask_norm":
                anomaly[
                    "median"
                ],

            "mad":
                anomaly[
                    "mad"
                ],

            "detected":
                detected,
        }
    )

    print(
        f"{model_name} | "
        f"Suspected class {suspected_class} "
        f"({CLASS_NAMES[suspected_class]}) | "
        f"Anomaly {suspected_score:.4f} | "
        f"Recovered ASR {suspected_result['reconstructed_asr']:.4f} | "
        f"Detected {detected}"
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

all_nc_df = pd.DataFrame(
    all_nc_rows
)

detection_df = pd.DataFrame(
    detection_rows
)

all_nc_df.to_csv(
    NC_ROOT
    / "all_models_summary.csv",
    index=False,
)

detection_df.to_csv(
    NC_ROOT
    / "detection_summary.csv",
    index=False,
)

display(
    detection_df
)

In [ ]:
for model_name in CHECKPOINTS:
    summary = pd.read_csv(
        NC_ROOT
        / model_name
        / "summary.csv"
    )

    model_figure_dir = (
        FIGURE_ROOT
        / model_name
    )

    model_figure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    plt.figure(
        figsize=(
            9,
            5,
        )
    )

    plt.bar(
        summary[
            "target_class"
        ],
        summary[
            "mask_norm"
        ],
    )

    plt.xticks(
        range(
            NUM_CLASSES
        ),
        CLASS_NAMES,
        rotation=45,
        ha="right",
    )

    plt.ylabel(
        "Mask L1 norm"
    )

    plt.title(
        f"{model_name}: Neural Cleanse mask norms"
    )

    plt.tight_layout()

    plt.savefig(
        model_figure_dir
        / "mask_norms.png",
        dpi=200,
    )

    plt.show()

    plt.figure(
        figsize=(
            9,
            5,
        )
    )

    plt.bar(
        summary[
            "target_class"
        ],
        summary[
            "anomaly_score"
        ],
    )

    plt.axhline(
        NC_ANOMALY_THRESHOLD,
        linestyle="--",
    )

    plt.xticks(
        range(
            NUM_CLASSES
        ),
        CLASS_NAMES,
        rotation=45,
        ha="right",
    )

    plt.ylabel(
        "Anomaly score"
    )

    plt.title(
        f"{model_name}: Neural Cleanse anomaly scores"
    )

    plt.tight_layout()

    plt.savefig(
        model_figure_dir
        / "anomaly_scores.png",
        dpi=200,
    )

    plt.show()

    figure, axes = plt.subplots(
        NUM_CLASSES,
        3,
        figsize=(
            8,
            24,
        ),
    )

    for class_id in range(
        NUM_CLASSES
    ):
        result = torch.load(
            NC_ROOT
            / model_name
            / f"class_{class_id}.pt",
            map_location="cpu",
        )

        mask = (
            result[
                "mask"
            ]
            .squeeze(
                0
            )
            .squeeze(
                0
            )
            .numpy()
        )

        pattern = (
            result[
                "pattern"
            ]
            .squeeze(
                0
            )
            .permute(
                1,
                2,
                0,
            )
            .numpy()
        )

        masked_pattern = (
            pattern
            * mask[
                ...,
                None,
            ]
        )

        axes[
            class_id,
            0,
        ].imshow(
            mask,
            cmap="gray",
            vmin=0,
            vmax=1,
        )

        axes[
            class_id,
            1,
        ].imshow(
            pattern,
            vmin=0,
            vmax=1,
        )

        axes[
            class_id,
            2,
        ].imshow(
            masked_pattern,
            vmin=0,
            vmax=1,
        )

        axes[
            class_id,
            0,
        ].set_ylabel(
            f"{class_id}: {CLASS_NAMES[class_id]}"
        )

        for column in range(
            3
        ):
            axes[
                class_id,
                column,
            ].set_xticks(
                []
            )

            axes[
                class_id,
                column,
            ].set_yticks(
                []
            )

    axes[
        0,
        0,
    ].set_title(
        "Mask"
    )

    axes[
        0,
        1,
    ].set_title(
        "Pattern"
    )

    axes[
        0,
        2,
    ].set_title(
        "Masked pattern"
    )

    plt.tight_layout()

    plt.savefig(
        model_figure_dir
        / "reconstructed_triggers.png",
        dpi=200,
    )

    plt.show()

target_comparison = all_nc_df[
    all_nc_df[
        "target_class"
    ]
    == TARGET_CLASS
][
    [
        "model",
        "mask_norm",
        "reconstructed_asr",
        "anomaly_score",
    ]
]

display(
    target_comparison
)

In [ ]:
archive_base = (
    REPO_ROOT
    / "adversarial_backdoor_neural_cleanse_artifacts"
)

archive_path = shutil.make_archive(
    str(
        archive_base
    ),
    "zip",
    root_dir=OUTPUT_ROOT,
)

print(
    "Created:",
    archive_path
)

try:
    from google.colab import files

    files.download(
        archive_path
    )

except ImportError:
    print(
        "Automatic download is only available in Colab."
    )